In [60]:
import numpy as np
import pandas as pd
import os

cols = ["Rndrng_NPI", "Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
        "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Stdzd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

# same cleaning as data_preprocessing.ipynb
keep = ((df["Rndrng_Prvdr_Cntry"] == "US")
        & (df["Tot_Srvcs"] >= df["Tot_Benes"])
        & (df["Tot_Bene_Day_Srvcs"] <= df["Tot_Srvcs"])
        & (df["Avg_Mdcr_Alowd_Amt"] > 0))
df = df[keep].reset_index(drop=True)
df["row_id"] = np.arange(len(df))
len(df) 

ERROR! Session/line number was not unique in database. History logging moved to new session 231


9781161

## Columns creation!

In [61]:
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))  # payment level (avoid log(0) = -inf for $0 amounts)
df["log_benes"] = np.log(df["Tot_Benes"])   # volume
df["srvcs_per_bene"] = np.log(df["Tot_Srvcs"] / df["Tot_Benes"]) # intensity
df["markup"] = np.log(df["Avg_Sbmtd_Chrg"].clip(lower=1e-6) / df["Avg_Mdcr_Alowd_Amt"])  # charge vs allowed

feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]   

# Peer groupping

In [62]:
MIN = 30 #peer group minimum size - I did some experiments with this and 30 seems to be a good balance between having enough peer groups and having enough members in each peer group.
levels = [["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],   # level 1
          ["HCPCS_Cd", "Place_Of_Srvc"], # level 2 (fallback)
          ["HCPCS_Cd"]] #level 3 (fallback)

df["peer_level"] = 0 # 0 = unscored, 1 = level 1, 2 = level 2, 3 = level 3
df["peer_id"] = -1 # no group id yet
df["peer_n"] = 0 # group size
for i, c in enumerate(levels, start=1):
    size = df.groupby(c, observed=True)["row_id"].transform("size")
    gid = df.groupby(c, observed=True).ngroup() + i * 10_000_000   # unique group id for each level (10M apart to avoid collisions)
    take = (df["peer_level"] == 0) & (size >= MIN)
    df.loc[take, "peer_id"] = gid[take]
    df.loc[take, "peer_n"] = size[take]
    df.loc[take, "peer_level"] = i

(df["peer_level"].value_counts(normalize=True) * 100).round(2)  # 0 = unscored

peer_level
1    97.63
2     2.03
0     0.24
3     0.10
Name: proportion, dtype: float64

# Robust z scores for our 5 columns

In [63]:
s = df[df["peer_level"]>0]
med = s.groupby("peer_id")[feats].transform("median") # median of each feature within each peer group
dev = (s[feats] - med).abs() # absolute deviation from median
mad = dev.groupby(s["peer_id"]).transform("median")  # median absolute deviation
scale = (1.4826 * mad).where(mad > 0, 1.2533 * dev.groupby(s["peer_id"]).transform("mean"))
rz = (s[feats] - med) / scale.where(scale > 0) # robust z-score (NaN if scale = 0)

rz_cols = [f"{f}_rz" for f in feats]
df[rz_cols] = rz.set_axis(rz_cols, axis=1) # rename columns, rows align by index (unscored -> NaN)

In [64]:
import os
os.makedirs("data/processed", exist_ok=True)

model_input = df[["row_id", "Rndrng_NPI", "peer_level", "peer_id", "peer_n"] + rz_cols].copy()
model_input[rz_cols] = model_input[rz_cols].astype("float32")      
model_input.to_parquet("data/processed/model_input.parquet", index=False)

## Explorations

In [65]:
model_input.head()

,row_id,Rndrng_NPI,peer_level,peer_id,peer_n,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz
0,0,1003000126,1,10026929,1213,-2.119651,0.920317,0.000000,0.501598
1,1,1003000126,1,10026931,18024,-1.696201,2.022970,0.000000,0.083158
2,2,1003000126,1,10026933,32360,-0.020690,0.251892,-0.674491,0.964575
3,3,1003000126,1,10026935,5592,-18.361212,-1.035447,-0.522472,0.230335
4,4,1003000126,1,10026937,35610,-3.948478,-0.515971,-0.413015,2.400614


In [66]:
model_input[model_input["peer_level"] == 0]["Rndrng_NPI"].nunique()

14702

In [67]:
df[df["peer_level"] == 0]["Rndrng_Prvdr_Type"].value_counts().head(10)

Rndrng_Prvdr_Type
Clinical Laboratory           4272
Orthopedic Surgery            1368
Ambulatory Surgical Center    1144
Diagnostic Radiology           905
General Surgery                886
Otolaryngology                 834
Ophthalmology                  801
Physician Assistant            708
Urology                        702
Nurse Practitioner             642
Name: count, dtype: int64

In [68]:
df['high_markup'] = df['markup_rz']>3.5

In [69]:
df[["high_markup"]].value_counts(normalize=True).mul(100).round(2)

high_markup
False          98.55
True            1.45
Name: proportion, dtype: float64

In [70]:
df.groupby("Place_Of_Srvc")["high_markup"].mean().mul(100).round(2)

C:\Users\talic\AppData\Local\Temp\ipykernel_23752\1830304588.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Place_Of_Srvc")["high_markup"].mean().mul(100).round(2)


Place_Of_Srvc
F    1.32
O    1.52
Name: high_markup, dtype: float64

In [71]:
df["chrg_to_alowd"] = df["Avg_Sbmtd_Chrg"] / df["Avg_Mdcr_Alowd_Amt"]
by_type = df.groupby("Rndrng_Prvdr_Type", observed = True).agg(
    rows = ("chrg_to_alowd", "size"),
   med_markup = ("chrg_to_alowd", "median")
)
by_type[by_type['rows']>10000].sort_values("med_markup", ascending=False).head(10)

,rows,med_markup
Rndrng_Prvdr_Type,,
Anesthesiology,203587,9.193096
Certified Registered Nurse Anesthetist (CRNA),115717,8.769750
Emergency Medicine,210968,5.852103
Independent Diagnostic Testing Facility (IDTF),31764,5.792212
Ambulatory Surgical Center,60937,4.728252
Diagnostic Radiology,1119939,4.355949
Interventional Radiology,59869,4.315429
Pathology,144616,4.151911
Pain Management,44627,4.074121


In [72]:
df["prov_high_rows"]= df.groupby("Rndrng_NPI")["high_markup"].transform("sum")

In [73]:
df[df["Rndrng_NPI"] == "1992998702"]

,Rndrng_NPI,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,Rndrng_Prvdr_Cntry,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,...,peer_level,peer_id,peer_n,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz,high_markup,chrg_to_alowd,prov_high_rows
9781074,1992998702,Centralized Flu,90480,O,US,144,152.0,152,41.838224,38.788487,...,1,10005288,9807,NaN,0.152956,-0.259734,554.258086,True,1.078625,2
9781075,1992998702,Centralized Flu,90653,O,US,214,214.0,214,83.490000,81.820000,...,1,10005289,9030,0.000000,1.434678,0.000000,-1.065809,False,1.020411,2
9781076,1992998702,Centralized Flu,90677,O,US,20,20.0,20,297.102000,291.161000,...,1,10005298,5684,0.803021,0.000000,0.000000,-1.223073,False,1.020405,2
9781077,1992998702,Centralized Flu,91320,O,US,71,73.0,73,151.483562,148.452603,...,1,10005306,7768,-0.043664,-0.358741,-0.674491,-1.388057,False,1.020417,2
9781078,1992998702,Centralized Flu,91322,O,US,69,74.0,74,159.311757,156.127838,...,1,10005307,4801,0.412278,-0.231368,0.359399,-0.675608,False,1.020393,2
9781079,1992998702,Centralized Flu,G0008,O,US,218,221.0,221,41.128597,30.945656,...,1,10005308,9617,0.000000,0.510017,1.072448,60.626469,True,1.329059,2
9781080,1992998702,Centralized Flu,G0009,O,US,20,20.0,20,31.248500,27.225000,...,1,10005309,5857,0.000000,0.000000,0.000000,3.057465,False,1.147787,2


In [74]:
results = df.groupby("Rndrng_NPI").agg(
    n_rows = ("high_markup", "size"),
    n_high = ("high_markup", "sum"),
    share_high = ("high_markup", "mean")
)
top = results[(results["n_rows"]>= 20) & (results["share_high"]>0.5)].sort_values("share_high", ascending=False)
types = df[["Rndrng_NPI", "Rndrng_Prvdr_Type"]].drop_duplicates()
top = top.merge(types, left_index=True, right_on="Rndrng_NPI")
top = top.reset_index(drop=True)

In [75]:
top

,n_rows,n_high,share_high,Rndrng_NPI,Rndrng_Prvdr_Type
0,20,19,0.950000,1053330647,General Surgery
1,23,21,0.913043,1124242029,Vascular Surgery
2,20,18,0.900000,1609971852,Diagnostic Radiology
3,24,21,0.875000,1699081117,General Surgery
4,23,20,0.869565,1699790931,Cardiology
...,...,...,...,...,...
240,45,23,0.511111,1821118811,Diagnostic Radiology
241,53,27,0.509434,1104208982,Diagnostic Radiology
242,55,28,0.509091,1053350298,Diagnostic Radiology
243,55,28,0.509091,1942232657,Diagnostic Radiology


In [76]:
eligible = results[results["n_rows"] >= 20].merge(types, left_index=True, right_on="Rndrng_NPI")
eligible["Rndrng_Prvdr_Type"].value_counts(normalize=True).head(5)   

Rndrng_Prvdr_Type
Diagnostic Radiology    0.220200
Cardiology              0.095590
Family Practice         0.089308
Internal Medicine       0.088281
Dermatology             0.041194
Name: proportion, dtype: float64

In [77]:
check = df[df["Rndrng_NPI"].isin(top["Rndrng_NPI"]) & df["high_markup"]]
check.groupby("Rndrng_Prvdr_Type")["Place_Of_Srvc"].value_counts(normalize=True).loc["Diagnostic Radiology"]

C:\Users\talic\AppData\Local\Temp\ipykernel_23752\2197345982.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  check.groupby("Rndrng_Prvdr_Type")["Place_Of_Srvc"].value_counts(normalize=True).loc["Diagnostic Radiology"]


Place_Of_Srvc
F    0.913116
O    0.086884
Name: proportion, dtype: float64

Markup is a noisy feature for some specialties (e.g. radiology), because of billing structure, not only provider behavior.
- Power BI: add a note for radiology: "markup flags may reflect billing structure".
- Models: if Isolation Forest also ranks many radiologists at the top, this explains why.

## Baseline model

In [78]:
z = df[rz_cols].clip(lower=0)   # high side only, NOT capped at 10 here
n_flags = (z > 3.5).sum(axis=1)

print(f"1+ feature > 3.5: {(n_flags >= 1).mean():.2%}")
print(f"2+ features > 3.5: {(n_flags >= 2).mean():.2%}")
print(f"one > 10 OR 2+ > 3.5: {((z.max(axis=1) > 10) | (n_flags >= 2)).mean():.2%}")

1+ feature > 3.5: 5.86%
2+ features > 3.5: 0.16%
one > 10 OR 2+ > 3.5: 1.19%


In [79]:
high = df[rz_cols].clip(lower=0)  # only above peers
scored = df["peer_level"] > 0

z_sorted = np.sort(high.fillna(0).to_numpy(), axis=1) # each row sorted ascending
top1, top2 = z_sorted[:, -1], z_sorted[:, -2]# largest and 2nd largest z

# anomaly: one feature > 10 OR two+ features > 3.5  <=>  baseline_raw > 3.5
df["baseline_raw"] = pd.Series(np.maximum(top2, top1 * 3.5 / 10), index=df.index).where(scored)

df["n_flags"] = (high > 3.5).sum(axis=1)
df["flag"] = df["baseline_raw"] > 3.5
df["score"] = (df["baseline_raw"].rank(pct=True) * 100).round(1)
df["risk_tier"] = "None"
fl = df["flag"]
df.loc[fl, "risk_tier"] = pd.cut(df.loc[fl, "baseline_raw"].rank(pct=True),
                                 [0, 0.5, 0.85, 1], labels=["Low", "Medium", "High"]).astype(str)
names = [c.replace("_rz", "") for c in rz_cols]
mask = (high > 3.5).to_numpy()
df["reason"] = [", ".join(n for n, m in zip(names, row) if m) for row in mask]

print(f"flagged: {df['flag'].mean():.2%}")                     
print(pd.crosstab(df["risk_tier"], df["flag"]))

flagged: 1.19%
flag         False  True 
risk_tier                
High             0  17398
Low              0  57992
Medium           0  40595
None       9665176      0


In [80]:
out = df[["row_id", "score", "flag", "reason", "risk_tier"]].copy()
out.insert(1, "method", "baseline_zscore")
out.to_parquet("data/processed/baseline_output.parquet", index=False)
out.head()

,row_id,method,score,flag,reason,risk_tier
0,0,baseline_zscore,53.8,False,,None
1,1,baseline_zscore,69.2,False,,None
2,2,baseline_zscore,39.0,False,,None
3,3,baseline_zscore,14.3,False,,None
4,4,baseline_zscore,76.3,False,,None


In [81]:
out.sort_values("score", ascending=False).head(10)

,row_id,method,score,flag,reason,risk_tier
3297693,3297693,baseline_zscore,100.0,True,markup,High
3297694,3297694,baseline_zscore,100.0,True,markup,High
3298090,3298090,baseline_zscore,100.0,True,markup,High
8251281,8251281,baseline_zscore,100.0,True,markup,High
1137242,1137242,baseline_zscore,100.0,True,markup,High
792111,792111,baseline_zscore,100.0,True,markup,High
791940,791940,baseline_zscore,100.0,True,markup,High
7254184,7254184,baseline_zscore,100.0,True,markup,High
5258170,5258170,baseline_zscore,100.0,True,markup,High
6589253,6589253,baseline_zscore,100.0,True,markup,High


In [82]:
df.sort_values("baseline_raw", ascending=False)[["row_id", "baseline_raw", "reason"] + rz_cols].head(10)

,row_id,baseline_raw,reason,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz
6326823,6326823,2.047549e+10,markup,0.0,-0.774811,-1.254000,5.850140e+10
640831,640831,1.748912e+10,markup,0.0,0.471130,2.618855,4.996891e+10
7919249,7919249,1.665401e+10,markup,0.0,0.021483,0.253557,4.758287e+10
1714115,1714115,1.665401e+10,markup,0.0,1.258588,1.089370,4.758287e+10
1703718,1703718,1.665401e+10,markup,0.0,1.163443,-0.423368,4.758287e+10
3389394,3389394,1.665401e+10,markup,0.0,1.127883,2.253458,4.758287e+10
2698091,2698091,1.665401e+10,markup,0.0,0.529332,-0.369106,4.758287e+10
8258214,8258214,1.665401e+10,markup,0.0,1.508386,0.545367,4.758287e+10
2491600,2491600,1.650667e+10,markup,0.0,0.357469,0.766171,4.716190e+10
1612516,1612516,1.618912e+10,markup,0.0,1.433740,-0.252622,4.625463e+10


In [83]:
# within-group correlation
s = df[df["peer_level"] > 0].copy()
s["log_chrg"] = np.log(s["Avg_Sbmtd_Chrg"].clip(lower=1e-6))
g = s.groupby("peer_id")
for c in ["log_chrg", "markup"]:
    s[c + "_w"] = s[c] - g[c].transform("median")      # deviation from own group median
print(s[["log_chrg_w", "markup_w"]].corr(method="spearman"))

            log_chrg_w  markup_w
log_chrg_w    1.000000  0.929602
markup_w      0.929602  1.000000


## Compararion of 30 vs. 100 min group sizes

In [84]:
f = "log_pay"
s = df[scored]

# group-level summary statistics
grp = s.groupby("peer_id").agg(
    n=(f, "size"),
    level=("peer_level", "first"),
    median=(f, "median"),
    mean=(f, "mean"),
    std=(f, "std"),
    q25=(f, lambda x: x.quantile(0.25)),
    q75=(f, lambda x: x.quantile(0.75)),
    min=(f, "min"),
    max=(f, "max"),
    flag_rate=("flag", "mean"),
)
grp["iqr"] = grp["q75"] - grp["q25"]

In [85]:
# summary statistics for peer groups of sizes 30-99
small = grp[(grp["n"] >= 30) & (grp["n"] < 100)]
small.describe().round(3)

,n,level,median,mean,std,q25,q75,min,max,flag_rate,iqr
count,7261.000,7261.000,7261.000,7261.000,7261.000,7261.000,7261.000,7261.000,7261.000,7261.000,7261.000
mean,54.769,1.262,3.747,3.722,0.170,3.650,3.826,3.226,3.949,0.024,0.176
std,19.498,0.458,1.667,1.645,0.259,1.638,1.685,1.717,1.737,0.054,0.404
min,30.000,1.000,-13.816,-8.196,0.000,-13.816,-4.638,-13.816,-4.605,0.000,0.000
25%,38.000,1.000,2.868,2.849,0.027,2.793,2.917,2.437,2.980,0.000,0.007
50%,50.000,1.000,3.814,3.788,0.091,3.700,3.875,3.347,3.961,0.000,0.070
75%,69.000,2.000,4.699,4.676,0.219,4.584,4.814,4.191,4.943,0.026,0.165
max,99.000,3.000,9.853,9.556,7.968,9.554,9.883,9.552,9.900,0.528,15.701


In [86]:
# summary statistics for peer groups of sizes 100+
big = grp[(grp["n"] >= 100)]
big.describe().round(3)

,n,level,median,mean,std,q25,q75,min,max,flag_rate,iqr
count,7480.000,7480.000,7480.000,7480.000,7480.000,7480.000,7480.000,7480.000,7480.000,7480.000,7480.000
mean,1247.896,1.069,3.656,3.623,0.144,3.570,3.709,2.709,3.828,0.014,0.139
std,3700.226,0.253,1.539,1.527,0.176,1.525,1.549,2.151,1.571,0.037,0.256
min,100.000,1.000,-4.660,-4.667,0.000,-4.711,-4.615,-13.816,-4.310,0.000,0.000
25%,159.000,1.000,2.824,2.800,0.036,2.748,2.891,2.029,2.949,0.000,0.013
50%,312.000,1.000,3.840,3.803,0.091,3.735,3.889,3.007,3.997,0.003,0.073
75%,839.000,1.000,4.575,4.552,0.181,4.497,4.629,3.892,4.756,0.014,0.134
max,91326.000,2.000,9.894,9.894,2.554,9.894,9.894,9.713,9.940,0.500,4.479


In [87]:
# Proportion of flagged claims in groups of size 100+
s.groupby(s["peer_n"] >= 100)["flag"].mean() * 100

peer_n
False    2.240653
True     1.151240
Name: flag, dtype: float64

## Baseline model as a function: comparing results across min group sizes

In [88]:
LEVELS = [
    ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],  # level 1
    ["HCPCS_Cd", "Place_Of_Srvc"],  # level 2
    ["HCPCS_Cd"],   # level 3
]
feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]
rz_cols = [f + "_rz" for f in feats]

# baseline model function (rewritten code from above to make it a function)
def run_baseline(df, min_size):
    out = df[["row_id"] + LEVELS[0] + feats].copy()
    out["peer_level"] = 0
    out["peer_id"] = -1
    out["peer_n"] = 0

    # 1. Peer groups 
    for i, keys in enumerate(LEVELS, start=1):
        free = out[out["peer_level"] == 0]
        size = free.groupby(keys, observed=True)["row_id"].transform("size")
        take = size.index[size >= min_size]
        gid = free.loc[take].groupby(keys, observed=True).ngroup()
        out.loc[take, "peer_level"] = i
        out.loc[take, "peer_id"] = gid + i * 10_000_000
        out.loc[take, "peer_n"] = size.loc[take]

    # 2. Robust z inside the groups
    s = out[out["peer_level"] > 0]
    med = s.groupby("peer_id")[feats].transform("median")
    dev = (s[feats] - med).abs()
    mad = dev.groupby(s["peer_id"]).transform("median")
    scale = (1.4826 * mad).where(mad > 0,
                                 1.2533 * dev.groupby(s["peer_id"]).transform("mean"))
    rz = (s[feats] - med) / scale.where(scale > 0)

    out[rz_cols] = np.nan
    out.loc[s.index, rz_cols] = rz.to_numpy()

    # 3. Combined rule: 1 features > 10 OR 2+ features > 3.5 (we are not looking ar the values lower)
    high = out[rz_cols].clip(lower=0).fillna(0).to_numpy()
    z_sorted = np.sort(high, axis=1)
    raw = np.maximum(z_sorted[:, -2], z_sorted[:, -1] * 3.5 / 10)
    out["baseline_raw"] = pd.Series(raw, index=out.index).where(out["peer_level"] > 0)
    out["flag"] = out["baseline_raw"] > 3.5

    return out[["row_id", "Rndrng_Prvdr_Type", "peer_level", "peer_id",
                "peer_n", "baseline_raw", "flag"] + rz_cols]

In [89]:
# run on a 30 and 100 min groups 
b30  = run_baseline(df, 30)
b100 = run_baseline(df, 100)
# Rows that change peers when min size goes from 30 to 100:
moved = (b30["peer_level"] == 1) & (b100["peer_level"] == 2)
print(moved.sum(), b30.loc[moved, "flag"].mean()*100, b100.loc[moved, "flag"].mean()*100)

243137 1.9474617191130927 1.8088567350917386


In [97]:
# Flag Rates
print(b30["flag"].mean()*100, b100["flag"].mean()*100)

1.1774062404248329 1.1591670968303251


In [98]:
# How rows change peer level between min 30 and min 100
display(pd.crosstab(b30["peer_level"], b100["peer_level"],
                    rownames=["Level (min 30)"], colnames=["Level (min 100)"]))

Level (min 100),0,1,2,3
Level (min 30),,,,
0,38704,0,3243,1888
1,44936,9249171,243137,12073
2,38352,0,134229,7954
3,5804,0,871,799


In [99]:
# Do both thresholds flag the same rows? (all data)
display(pd.crosstab(b30["flag"], b100["flag"],
                    rownames=["Flag (min 30)"], colnames=["Flag (min 100)"]))


Flag (min 100),False,True
Flag (min 30),,
False,9660811,5186
True,6970,108194


In [100]:
# Same check, only rows that moved from level 1 to level 2
display(pd.crosstab(b30.loc[moved, "flag"], b100.loc[moved, "flag"],
                    rownames=["Flag (min 30)"], colnames=["Flag (min 100)"]))

Flag (min 100),False,True
Flag (min 30),,
False,235777,2625
True,2962,1773


In [101]:
f30  = b30.loc[moved, "flag"]
f100 = b100.loc[moved, "flag"]

both  = (f30 & f100).sum()
union = (f30 | f100).sum()
print(f"Flagged under both: {both:,}")
print(f"Flagged under at least one: {union:,}")

Flagged under both: 1,773
Flagged under at least one: 7,360


Among moved rows, 4,735 are flagged at min 30 and 4,398 at min 100, but only 1,773 are flagged under both. 
The flag rate is the same, but more than half of the flags depend on the threshold.

In [102]:
# Flagged under one threshold but not the other
t = "Rndrng_Prvdr_Type"
side = pd.concat([
    b100.loc[only100, t].value_counts().rename("Only at min 100"),
    b30.loc[only30, t].value_counts().rename("Only at min 30"),
], axis=1).fillna(0).astype(int)

side.sort_values("Only at min 100", ascending=False).head(10)

,Only at min 100,Only at min 30
Rndrng_Prvdr_Type,,
Ambulatory Surgical Center,815,85
Nurse Practitioner,149,267
Family Practice,112,91
Portable X-Ray Supplier,89,22
Hand Surgery,86,9
Internal Medicine,81,165
Hematology-Oncology,77,78
Clinical Laboratory,72,97
Cardiology,63,99


In [103]:
asc = only100 & (b100["Rndrng_Prvdr_Type"] == "Ambulatory Surgical Center")
(b100.loc[asc, rz_cols] > 3.5).mean().round(2)

log_pay_rz           0.97
log_benes_rz         0.04
srvcs_per_bene_rz    0.06
markup_rz            0.02
dtype: float64

**Findings:**
Flags that appear only at min 100 relates to the provider types payment structure: Ambulatory Surgical Centers (815 vs 85), Hand Surgery (86 vs 9), Interventional Pain Management (62 vs 3) and Portable X-Ray (89 vs 22). With min 100 these providers fall back into mixed groups and get compared with physicians, so they look abnormal even when they aren't. Min 30 keeps them compared to their own.